# Lesson 2.6 — Load the dataset and build batches

This notebook covers the step between "the dataset exists on disk" and "the model
can consume it":

`LeRobotDataset` → `dataset[index]` → `DataLoader` → batch tensors.

It is deliberately mechanical. Nothing is learned here; the point is to know the
exact shape, dtype, and meaning of what enters a policy.


## 2.6.1 — Read the dataset and one frame

`dataset[index]` returns a single timestep: `observation.state [42]` and
`action [8]`, both `float32`, plus the index and timestamp bookkeeping fields.

The assertions pin those shapes so a schema change fails loudly instead of
silently training on the wrong thing.


In [1]:
import sys
print(sys.executable)

/home/bowenyuan/miniforge3/envs/embodied/bin/python


In [2]:
from pathlib import Path

print("Current working directory:", Path.cwd())

Current working directory: /home/bowenyuan/Projects/embodied-ai-learning/notebooks


In [3]:
from pathlib import Path
from lerobot.datasets.lerobot_dataset import LeRobotDataset

cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATASET_ROOT = PROJECT_ROOT / "datasets" / "lerobot" / "pickcube"

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())
print("Data exists:", (DATASET_ROOT / "data").exists())
print("Meta exists:", (DATASET_ROOT / "meta").exists())

dataset = LeRobotDataset(
    repo_id="pickcube",
    root=DATASET_ROOT,
)

print("Dataset length:", len(dataset))
print("Episodes:", dataset.num_episodes)
print("FPS:", dataset.fps)

Project root: /home/bowenyuan/Projects/embodied-ai-learning
Dataset root: /home/bowenyuan/Projects/embodied-ai-learning/datasets/lerobot/pickcube
Dataset exists: True
Data exists: True
Meta exists: True
Dataset length: 50
Episodes: 1
FPS: 50


In [4]:
sample = dataset[0]

print("Sample type:", type(sample))
print("\nKeys:")

for key, value in sample.items():
    shape = getattr(value, "shape", None)
    dtype = getattr(value, "dtype", type(value))
    print(f"{key:30s} shape={str(shape):15s} dtype={dtype}")

Sample type: <class 'dict'>

Keys:
observation.state              shape=torch.Size([42]) dtype=torch.float32
action                         shape=torch.Size([8]) dtype=torch.float32
timestamp                      shape=torch.Size([])  dtype=torch.float32
frame_index                    shape=torch.Size([])  dtype=torch.int64
episode_index                  shape=torch.Size([])  dtype=torch.int64
index                          shape=torch.Size([])  dtype=torch.int64
task_index                     shape=torch.Size([])  dtype=torch.int64
task                           shape=None            dtype=<class 'str'>


In [5]:
state = sample["observation.state"]
action = sample["action"]

print("\nState shape:", state.shape)
print("Action shape:", action.shape)

print("\nState:")
print(state)

print("\nAction:")
print(action)


State shape: torch.Size([42])
Action shape: torch.Size([8])

State:
tensor([ 3.5281e-02,  4.0070e-01,  1.9575e-02, -1.9187e+00,  3.7351e-02,
         2.3366e+00,  8.0440e-01,  4.0000e-02,  4.0000e-02,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.2254e-02,
         3.8011e-02,  1.8215e-01, -1.7689e-02,  9.9980e-01,  4.2880e-03,
         7.9842e-03,  2.6816e-02, -1.9813e-03,  2.8893e-01, -7.4868e-04,
         5.3644e-02,  2.0000e-02,  5.6876e-01,  0.0000e+00,  0.0000e+00,
         8.2250e-01, -1.3002e-02,  1.5633e-02, -1.6215e-01,  2.7564e-02,
        -5.5626e-02,  2.6893e-01])

Action:
tensor([ 0.1791,  0.9838,  0.1854, -0.8309, -0.2816, -0.4202, -0.8190,  0.0052])


In [6]:
assert state.shape == (42,)
assert action.shape == (8,)
assert len(dataset) == 50
assert dataset.num_episodes == 1
assert dataset.fps == 50

print("Single-frame inspection: PASS")

Single-frame inspection: PASS


## 2.6.2 — Batch the samples

The `DataLoader` stacks samples into a batch, so the policy receives
`states [8,42]` and `target_actions [8,8]`. The new leading dimension is the batch
dimension; per-sample semantics are unchanged.

Note `shuffle=True`: with a single trajectory this mixes adjacent frames across
batches. Acceptable for a single-frame MLP, and exactly what must not be reused
once temporal windows or multiple episodes exist.

The `index` / `episode_index` / `frame_index` fields in the batch are what let a
row be traced back to its episode and time, which matters as soon as more than
one trajectory exists.


In [5]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

In [6]:
batch = next(iter(dataloader))

print("Batch type:", type(batch))
print("\nBatch fields:")

for key, value in batch.items():
    if hasattr(value, "shape"):
        print(
            f"{key:30s}",
            f"shape={str(value.shape):15s}",
            f"dtype={value.dtype}",
        )
    else:
        print(
            f"{key:30s}",
            f"type={type(value)}",
            f"value={value}",
        )

Batch type: <class 'dict'>

Batch fields:
observation.state              shape=torch.Size([8, 42]) dtype=torch.float32
action                         shape=torch.Size([8, 8]) dtype=torch.float32
timestamp                      shape=torch.Size([8]) dtype=torch.float32
frame_index                    shape=torch.Size([8]) dtype=torch.int64
episode_index                  shape=torch.Size([8]) dtype=torch.int64
index                          shape=torch.Size([8]) dtype=torch.int64
task_index                     shape=torch.Size([8]) dtype=torch.int64
task                           type=<class 'list'> value=['pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube']


In [7]:
states = batch["observation.state"]
actions = batch["action"]

print("States:", states.shape)
print("Actions:", actions.shape)

assert states.shape == (8, 42)
assert actions.shape == (8, 8)

print("Batch inspection: PASS")

States: torch.Size([8, 42])
Actions: torch.Size([8, 8])
Batch inspection: PASS


In [8]:
print("Global indices:", batch["index"])
print("Episode indices:", batch["episode_index"])
print("Frame indices:", batch["frame_index"])
print("Timestamps:", batch["timestamp"])
print("Tasks:", batch["task"])

Global indices: tensor([0, 1, 2, 3, 4, 5, 6, 7])
Episode indices: tensor([0, 0, 0, 0, 0, 0, 0, 0])
Frame indices: tensor([0, 1, 2, 3, 4, 5, 6, 7])
Timestamps: tensor([0.0000, 0.0200, 0.0400, 0.0600, 0.0800, 0.1000, 0.1200, 0.1400])
Tasks: ['pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube']
